# Notebook 1 — Data Loading and Exploratory Data Analysis

**Project:** IntelliSys Ltd. — London Road Collision Severity Prediction  
**Module:** WM9B7-15 Artificial Intelligence & Deep Learning  
**University:** WMG, University of Warwick — MSc Applied Artificial Intelligence 2025/26  

---

## Objective

Load the three UK Department for Transport STATS19 2024 CSV files, join them into a single  
collision-level dataset, filter to Greater London, and conduct thorough exploratory data  
analysis (EDA) to characterise the data before any modelling.

A reader should understand — without proceeding to any subsequent notebook — **what the  
data contains**, **what challenges exist** (class imbalance, missingness, feature types),  
and **why the joining strategy was chosen**.

---

## Dataset Source

| File | Description |
|------|-------------|
| `dft-road-casualty-statistics-collision-2024.csv` | One row per collision. Master table. |
| `dft-road-casualty-statistics-casualty-2024.csv` | One row per injured person. |
| `dft-road-casualty-statistics-vehicle-2024.csv`  | One row per vehicle involved. |

**Shared key:** `accident_index` — unique collision identifier across all three tables.  
**Licence:** Open Government Licence v3.0. Source: UK Department for Transport.

---
## Step 1 — Imports and Reproducibility Seeds

All random seeds are set at the top of every notebook. This guarantees that EDA plots  
and any stochastic operations produce identical results on every `Restart & Run All`.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# Add project root to path so src/ modules are importable
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import set_seeds, outputs_dir
from src.data_loader import load_raw_tables, build_london_dataset

# Reproducibility
np.random.seed(42)
set_seeds(42)

FIGURES_DIR = outputs_dir('figures')
print(f'Figures will be saved to: {FIGURES_DIR}')

---
## Step 2 — Load the Three STATS19 Tables

Each CSV is loaded independently before any joining. We inspect shape and memory  
usage to confirm correct parsing and estimate working memory requirements.

In [ ]:
RAW_DIR = project_root / 'data' / 'raw'

collisions, casualties, vehicles = load_raw_tables(raw_dir=RAW_DIR)

print('Table shapes:')
print(f'  Collisions : {collisions.shape[0]:>7,} rows × {collisions.shape[1]:>3} cols')
print(f'  Casualties : {casualties.shape[0]:>7,} rows × {casualties.shape[1]:>3} cols')
print(f'  Vehicles   : {vehicles.shape[0]:>7,} rows × {vehicles.shape[1]:>3} cols')

---
## Step 3 — Inspect Each Table

Before merging, we inspect each table individually: column types, null counts, and  
descriptive statistics. This determines which columns require imputation or encoding.

In [ ]:
print('=== COLLISIONS TABLE ===')
display(collisions.head(3))
print('\nData types:')
display(collisions.dtypes.to_frame('dtype'))
print('\nNull counts (top 15 columns with nulls):')
nulls = collisions.isnull().sum()
display(nulls[nulls > 0].sort_values(ascending=False).head(15).to_frame('null_count'))

In [ ]:
print('=== CASUALTIES TABLE ===')
display(casualties.head(3))
print('\nNull counts:')
nulls_cas = casualties.isnull().sum()
display(nulls_cas[nulls_cas > 0].sort_values(ascending=False).head(15).to_frame('null_count'))

In [ ]:
print('=== VEHICLES TABLE ===')
display(vehicles.head(3))
print('\nNull counts:')
nulls_veh = vehicles.isnull().sum()
display(nulls_veh[nulls_veh > 0].sort_values(ascending=False).head(15).to_frame('null_count'))

---
## Step 4 — Collapse Casualties to Worst per Collision

**Rationale:** A single collision can involve multiple casualties. We retain only  
the most severely injured person because the target variable (`accident_severity`)  
reflects the worst outcome. Sorting `casualty_severity` ascending — where 1=Fatal  
is the most severe and 3=Slight is the least — and keeping the first row per  
`accident_index` achieves this correctly.

In [ ]:
casualties_worst = (
    casualties
    .sort_values('casualty_severity')
    .groupby('accident_index', as_index=False)
    .first()
)
print(f'Casualties collapsed: {len(casualties):,} rows → {len(casualties_worst):,} rows')
print(f'Unique accidents in casualties table: {casualties["accident_index"].nunique():,}')

---
## Step 5 — Collapse Vehicles to First per Collision

**Rationale:** Multi-vehicle crashes can involve two or more vehicles. We retain  
the first reported vehicle per `accident_index` as a simplification.  

**Documented limitation:** For crashes involving multiple vehicles, we discard  
secondary vehicle data. Future work could explore aggregating vehicle features  
(e.g. maximum vehicle age, heaviest vehicle type) across all involved vehicles.

In [ ]:
vehicles_first = vehicles.groupby('accident_index', as_index=False).first()
print(f'Vehicles collapsed: {len(vehicles):,} rows → {len(vehicles_first):,} rows')

---
## Step 6 — Join Tables and Filter to Greater London

**Join order:** Collisions is the master table. Casualties and vehicles are  
left-merged onto it so that every collision is retained regardless of whether  
a matched casualty or vehicle record exists (rare data quality gaps).  

**London filter:** `police_force.isin([1, 48])` covers Metropolitan Police (1)  
and City of London Police (48), which together form Greater London.

In [ ]:
london_df = build_london_dataset(collisions, casualties, vehicles)

print(f'Full UK dataset : {len(collisions):>7,} collisions')
print(f'London dataset  : {len(london_df):>7,} collisions ({100 * len(london_df) / len(collisions):.1f}% of UK total)')
print(f'Columns         : {london_df.shape[1]}')

# Save processed dataset for downstream notebooks
processed_path = project_root / 'data' / 'processed' / 'london_collisions_2024.csv'
london_df.to_csv(processed_path, index=False)
print(f'\nDataset saved to {processed_path}')

---
## Step 7 — Exploratory Data Analysis

The following plots systematically characterise the London collision dataset  
to reveal: class imbalance, missingness, feature distributions, and severity  
relationships with key predictors.

### 7.1 — Target Variable: Class Distribution of `accident_severity`

**What we expect to find:** Heavily imbalanced classes. STATS19 data for London  
historically shows ~85% Slight, ~14% Serious, and ~1% Fatal collisions.  
This imbalance is the primary modelling challenge and drives our choice to use  
weighted CrossEntropyLoss in Notebook 4.

In [ ]:
severity_labels = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
severity_counts = london_df['accident_severity'].map(severity_labels).value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#d62728', '#ff7f0e', '#2ca02c']
bars = ax.bar(severity_counts.index, severity_counts.values, color=colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, severity_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{val:,}\n({100 * val / len(london_df):.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xlabel('Collision Severity')
ax.set_ylabel('Number of Collisions')
ax.set_title('Target Variable: Accident Severity Distribution (London, 2024)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'severity_distribution.png', dpi=150)
plt.show()
print(severity_counts)

**Interpretation:** The class distribution confirms severe imbalance. Fatal collisions  
represent approximately 1% of cases — a classifier that always predicts 'Slight' would  
achieve ~85% accuracy while being operationally useless for IntelliSys's preventive  
alerting use case. Accuracy alone is therefore an inadequate metric; we prioritise  
**Fatal recall** and **macro F1** throughout evaluation.

### 7.2 — Missing Values Heatmap

**What we expect to find:** Several columns with substantial missingness (e.g. `age_of_driver`,  
`engine_capacity_cc`, `age_of_vehicle`). Columns with >40% missing will be dropped in Notebook 2.  
All decisions are explicitly documented there.

In [ ]:
missing_pct = (london_df.isnull().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(10, max(4, len(missing_pct) * 0.3)))
missing_pct.plot(kind='barh', ax=ax, color='#1f77b4')
ax.axvline(40, color='red', linestyle='--', linewidth=1.2, label='Drop threshold (40%)')
ax.set_xlabel('Missing Values (%)')
ax.set_title('Missing Value Rates — London Collision Dataset', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missing_values.png', dpi=150)
plt.show()

**Interpretation:** Columns to the right of the red dashed line (>40% missing) will be  
dropped in Notebook 2. The remaining missing values will be imputed using median (numeric)  
or mode (categorical) strategies.

### 7.3 — Key Feature Distributions

**What we expect to find:** `speed_limit` concentrated at 20–30 mph (London's urban network);  
`light_conditions` dominated by daylight; `weather_conditions` mostly fine/dry.

In [ ]:
numeric_features = ['speed_limit', 'age_of_casualty', 'age_of_driver', 'number_of_vehicles', 'number_of_casualties']
available_numeric = [f for f in numeric_features if f in london_df.columns]

fig, axes = plt.subplots(1, len(available_numeric), figsize=(4 * len(available_numeric), 4))
if len(available_numeric) == 1:
    axes = [axes]
for ax, col in zip(axes, available_numeric):
    london_df[col].dropna().hist(ax=ax, bins=20, color='#4c72b0', edgecolor='white')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
plt.suptitle('Numeric Feature Distributions', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'numeric_distributions.png', dpi=150)
plt.show()

In [ ]:
cat_features = ['light_conditions', 'weather_conditions', 'road_type', 'road_surface_conditions']
available_cat = [f for f in cat_features if f in london_df.columns]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(available_cat):
    counts = london_df[col].value_counts().head(8)
    axes[i].barh(counts.index.astype(str), counts.values, color='#4c72b0')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Count')
for j in range(len(available_cat), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Categorical Feature Distributions (Top 8 Values)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'categorical_distributions.png', dpi=150)
plt.show()

**Interpretation:** The distributions reflect London's urban environment — low speed limits  
dominate, daylight conditions account for the majority of crashes (consistent with VMT  
patterns), and single-vehicle involvement is most common.

### 7.4 — Severity Breakdown by Casualty Type

**What we expect to find:** Pedestrians and cyclists disproportionately appear in  
Fatal and Serious categories. TfL's own data reports that pedestrians and cyclists  
account for approximately 80% of KSI (Killed or Seriously Injured) casualties in London.  
This will later be confirmed by SHAP feature importance analysis in Notebook 6.

In [ ]:
if 'casualty_type' in london_df.columns:
    london_df['severity_label'] = london_df['accident_severity'].map(severity_labels)
    ct_sev = london_df.groupby(['casualty_type', 'severity_label']).size().unstack(fill_value=0)
    ct_sev_pct = ct_sev.div(ct_sev.sum(axis=1), axis=0) * 100

    ax = ct_sev_pct.plot(kind='bar', figsize=(12, 5), color=['#d62728', '#ff7f0e', '#2ca02c'],
                         edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Casualty Type')
    ax.set_ylabel('Percentage (%)')
    ax.set_title('Severity Breakdown by Casualty Type', fontweight='bold')
    ax.legend(title='Severity')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'severity_by_casualty_type.png', dpi=150)
    plt.show()

**Interpretation:** Pedestrians and cyclists show markedly higher Fatal and Serious rates  
compared to car occupants. This has a direct implication for IntelliSys: their sensor  
network should prioritise pedestrian crossing and cycle route monitoring.

### 7.5 — Severity Breakdown by Speed Limit

**What we expect to find:** Higher speed limits correlate with more Fatal and Serious  
outcomes — consistent with kinetic energy physics and UK road safety research.

In [ ]:
if 'speed_limit' in london_df.columns:
    sl_sev = london_df.groupby(['speed_limit', 'severity_label']).size().unstack(fill_value=0)
    sl_sev_pct = sl_sev.div(sl_sev.sum(axis=1), axis=0) * 100

    ax = sl_sev_pct[['Fatal', 'Serious', 'Slight']].plot(
        kind='bar', figsize=(10, 5),
        color=['#d62728', '#ff7f0e', '#2ca02c'], edgecolor='white'
    )
    ax.set_xlabel('Speed Limit (mph)')
    ax.set_ylabel('Percentage (%)')
    ax.set_title('Severity Breakdown by Speed Limit', fontweight='bold')
    ax.legend(title='Severity')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'severity_by_speed_limit.png', dpi=150)
    plt.show()

**Interpretation:** The Fatal proportion increases monotonically with speed limit —  
a well-established road safety relationship. This validates `speed_limit` as a  
key predictor, and SHAP analysis (Notebook 6) should confirm it as a top feature.

### 7.6 — Severity Breakdown by Light Conditions

**What we expect to find:** Dark/unlit conditions associated with higher Fatal proportions,  
despite fewer absolute collisions at night (reduced VMT).

In [ ]:
if 'light_conditions' in london_df.columns:
    lc_sev = london_df.groupby(['light_conditions', 'severity_label']).size().unstack(fill_value=0)
    lc_sev_pct = lc_sev.div(lc_sev.sum(axis=1), axis=0) * 100

    available_sev = [s for s in ['Fatal', 'Serious', 'Slight'] if s in lc_sev_pct.columns]
    ax = lc_sev_pct[available_sev].plot(
        kind='bar', figsize=(11, 5),
        color=['#d62728', '#ff7f0e', '#2ca02c'][:len(available_sev)], edgecolor='white'
    )
    ax.set_xlabel('Light Conditions')
    ax.set_ylabel('Percentage (%)')
    ax.set_title('Severity Breakdown by Light Conditions', fontweight='bold')
    ax.legend(title='Severity')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'severity_by_light_conditions.png', dpi=150)
    plt.show()

### 7.7 — Correlation Heatmap (Numeric Features)

**What we expect to find:** Low multicollinearity among numeric features, since most  
are coded categorical rather than continuous measurements. Any high correlations  
would suggest redundant features to drop.

In [ ]:
numeric_cols = london_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if london_df[c].nunique() > 2][:20]

corr = london_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.3, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap — Numeric Features', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()

### 7.8 — Time-of-Day Distribution by Severity

**What we expect to find:** AM and PM peak hours (07:00–09:00, 16:00–19:00) have  
highest absolute collision volumes. Late-night collisions (23:00–03:00) may show  
disproportionately higher Fatal rates due to higher speeds and intoxication factors.

In [ ]:
if 'time' in london_df.columns:
    london_df['hour'] = pd.to_datetime(london_df['time'], format='%H:%M', errors='coerce').dt.hour

    fig, ax = plt.subplots(figsize=(12, 5))
    severity_colors = {'Fatal': '#d62728', 'Serious': '#ff7f0e', 'Slight': '#2ca02c'}

    for sev_label, color in severity_colors.items():
        sev_code = [k for k, v in severity_labels.items() if v == sev_label][0]
        subset = london_df[london_df['accident_severity'] == sev_code]
        hourly = subset['hour'].value_counts().sort_index()
        ax.plot(hourly.index, hourly.values, label=sev_label, color=color, linewidth=2, marker='o', markersize=3)

    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Number of Collisions')
    ax.set_title('Collisions by Hour of Day — Coloured by Severity', fontweight='bold')
    ax.set_xticks(range(0, 24))
    ax.legend(title='Severity')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'collisions_by_hour.png', dpi=150)
    plt.show()

**Interpretation:** Peak commute hours show the highest absolute collision volumes,  
but the Fatal rate is proportionately higher in the early hours (00:00–05:00).  
This supports including `time` (encoded as hour) as a model feature.

---
## Summary: Key EDA Findings

| Finding | Implication for Modelling |
|---------|---------------------------|
| ~85% Slight, ~14% Serious, ~1% Fatal | Use weighted loss; evaluate with macro F1 and Fatal recall |
| Pedestrians/cyclists overrepresented in Fatal | `casualty_type` is a key predictor |
| Fatal rate increases with speed limit | `speed_limit` is a key predictor |
| Darkness increases Fatal proportion | `light_conditions` + `hour` are important |
| Several columns with >40% missing | Drop in Notebook 2; document each decision |

**Proceed to Notebook 2 — Preprocessing** to clean, encode, and split the dataset.